# KAIL — Notebook 01: Baseline Evaluation

> **Kora AI Lab** | H-AZR Research Pipeline  
> https://github.com/kora-ai-lab/h-azr

## What this notebook does

**Scenario A** — Base model, no training.

1. Load `Qwen2.5-Coder-1.5B-Instruct` in 4-bit on T4
2. Run inference on TruthfulQA and HumanEval
3. Measure **hallucination rate** and **code accuracy**
4. Save baseline metrics → used to compare Scenarios B, C, D

**Runtime:** ~25 min on free Colab T4  
**VRAM:** ~6GB

In [ ]:
# @title Setup — Install dependencies
!pip install -q transformers>=4.40.0 bitsandbytes>=0.43.0 accelerate>=0.28.0 \
              datasets>=2.18.0 evaluate peft tqdm

In [ ]:
# @title Imports
import torch
import numpy as np
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.notebook import tqdm

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title Load model in 4-bit (QLoRA-ready)
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# @title Helper: generate response
def generate(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# @title Quick sanity check
response = generate("Write a Python function that returns the nth Fibonacci number.")
print(response)

## Part 1 — Hallucination evaluation (TruthfulQA subset)

In [ ]:
# @title Load TruthfulQA
dataset = load_dataset("truthful_qa", "generation", split="validation")
# Use first 100 for speed on free Colab
questions = [ex["question"] for ex in dataset.select(range(100))]
best_answers = [ex["best_answer"] for ex in dataset.select(range(100))]
print(f"Loaded {len(questions)} TruthfulQA questions")
print(f"\nExample: {questions[0]}")
print(f"Best answer: {best_answers[0]}")

In [ ]:
# @title Run baseline eval on TruthfulQA
results = []

for q, ans in tqdm(zip(questions, best_answers), total=len(questions), desc="Evaluating"):
    pred = generate(q, max_new_tokens=128)
    is_correct = ans.lower()[:30] in pred.lower()
    results.append({
        "question": q,
        "best_answer": ans,
        "prediction": pred,
        "correct": is_correct,
    })

accuracy = sum(r["correct"] for r in results) / len(results)
hallucination_rate = 1.0 - accuracy

print(f"\n{'='*50}")
print(f"SCENARIO A — BASELINE RESULTS (TruthfulQA)")
print(f"{'='*50}")
print(f"Accuracy:           {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f"Hallucination rate: {hallucination_rate:.3f} ({hallucination_rate*100:.1f}%)")
print(f"n samples:          {len(results)}")

## Part 2 — Code evaluation (HumanEval subset)

In [ ]:
# @title Simple code execution test
import subprocess, tempfile, os

def execute_code(code: str, timeout: int = 5) -> bool:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        r = subprocess.run(["python", tmp], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except:
        return False
    finally:
        os.unlink(tmp)

CODE_TASKS = [
    {
        "prompt": "Write a Python function `add(a, b)` that returns the sum of two numbers.",
        "test": "assert add(2, 3) == 5\nassert add(-1, 1) == 0",
    },
    {
        "prompt": "Write a Python function `is_palindrome(s)` that returns True if s is a palindrome.",
        "test": "assert is_palindrome('racecar') == True\nassert is_palindrome('hello') == False",
    },
    {
        "prompt": "Write a Python function `fibonacci(n)` that returns the nth Fibonacci number (0-indexed).",
        "test": "assert fibonacci(0) == 0\nassert fibonacci(1) == 1\nassert fibonacci(6) == 8",
    },
    {
        "prompt": "Write a Python function `count_vowels(s)` that returns the number of vowels in string s.",
        "test": "assert count_vowels('hello') == 2\nassert count_vowels('rhythm') == 0",
    },
    {
        "prompt": "Write a Python function `flatten(lst)` that flattens a list of lists.",
        "test": "assert flatten([[1,2],[3,4]]) == [1,2,3,4]\nassert flatten([[1],[2],[3]]) == [1,2,3]",
    },
]

code_results = []
for task in tqdm(CODE_TASKS, desc="Code eval"):
    code = generate(task["prompt"], max_new_tokens=256)
    # Strip markdown fences if present
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    elif "```" in code:
        code = code.split("```")[1].split("```")[0]
    full_code = code + "\n\n" + task["test"]
    passed = execute_code(full_code)
    code_results.append({"task": task["prompt"][:50], "passed": passed, "code": code})

code_pass_rate = sum(r["passed"] for r in code_results) / len(code_results)
print(f"\nCode pass rate: {code_pass_rate:.2f} ({code_pass_rate*100:.0f}%)")

In [ ]:
# @title Save baseline metrics
baseline_metrics = {
    "scenario": "A_baseline",
    "model": MODEL_NAME,
    "truthfulqa": {
        "accuracy": accuracy,
        "hallucination_rate": hallucination_rate,
        "n_samples": len(results),
    },
    "code_eval": {
        "pass_rate": code_pass_rate,
        "n_tasks": len(code_results),
    }
}

with open("baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print("Metrics saved to baseline_metrics.json")
print(json.dumps(baseline_metrics, indent=2))

## Summary

**Scenario A complete.** You now have baseline metrics.

**Next step:** Run `02_h_neuron_probe.ipynb` to identify the H-Neurons in this model.  
These will be used as the penalty signal in Stage 3 (H-AZR).

---
*KAIL — Kora AI Lab | github.com/kora-ai-lab*